# Allele Primer Design Pipeline

Design RNA-sense gene-specific primers near informative B6/Cast SNPs to pair with an RT/poly-dT primer. The pipeline reports progress for every gene and SNP target and writes per-gene plus consolidated outputs.

In [ ]:
# Configuration: edit parameters only in this cell.
import os
from pathlib import Path

GENES = ["Xist", "Tsix", "Kdm5c", "Kdm6a", "Mid1", "Rlim", "Rbmx"]
OUTPUT_DIR = Path("/Users/gmgao/Dropbox/Caltech_PostDoc_GuttmanLab/constructs_primers_FISHprobes/qPCR LibAmp PCR primers/shortreadSeq-X genes set to pair RTprimer")

CONFIG = {
    # Reference resources and samples
    "gtf_path": "/Volumes/guttman/genomes/mm10/annotation/mm10.refGene.gtf.gz",
    "genome_fasta": "/Volumes/guttman/genomes/mm10/fasta/mm10.fa",
    "snp_vcf": "/Volumes/guttman/genomes/mm10/variants/mgp.v5.merged.snps_all.dbSNP142.vcf.gz",
    "blast_database": "/Volumes/guttman/genomes/mm10/fasta/blastdb/mm10_blastdb",
    "vcf_b6_sample": "C57BL_6NJ",
    "vcf_cast_sample": "CAST_EiJ",
    "curated_transcript_prefixes": ("NM_", "NR_"),
    # External tools
    "conda_env": "bioinfo",
    "samtools": "/opt/miniconda3/envs/bioinfo/bin/samtools",
    "blastn": "/opt/miniconda3/envs/bioinfo/bin/blastn",
    "primer3_core": "/opt/miniconda3/envs/bioinfo/bin/primer3_core",
    # Target selection and final oligo
    "handle": "2PUNI",  # 2PUNI or 2PBC
    "top_snps": 5,
    "snp_window_toward_polydt": 100,
    # Primer3 and post-filter settings
    "primer3_num_return": 50,
    "primer_min_size": 18,
    "primer_opt_size": 22,
    "primer_max_size": 30,
    "valid_primer_length_min": 18,
    "valid_primer_length_max": 30,
    "primer_min_tm": 55.0,
    "primer_opt_tm": 60.0,
    "primer_max_tm": 65.0,
    "post_filter_tm_min": 55.0,
    "post_filter_tm_max": 65.0,
    "primer_min_gc": 40.0,
    "primer_max_gc": 60.0,
    "post_filter_gc_min": 40.0,
    "post_filter_gc_max": 60.0,
    "primer_gc_clamp": 1,
    "primer_max_poly_x": 4,
    "primer_max_self_any": 8.0,
    "primer_max_self_end": 3.0,
    "max_homopolymer_run": 3,
    "max_quad_g_run": 3,
    # BLAST specificity
    "blast_specificity": True,
    "blast_candidates_per_snp": 5,
    "blast_min_unique_len": 18,
    "blast_min_unique_identity": 95.0,
    "blast_noise_len": 12,
    # Rich progress display
    "show_progress": True,
}

# Automated validation mode; production defaults above remain unchanged.
if os.environ.get("ALLELE_PRIMER_SMOKE_TEST") == "1":
    GENES = ["Xist", "Kdm5c"]
    OUTPUT_DIR = Path("/private/tmp/allele_primer_notebook_smoke")
    CONFIG["blast_specificity"] = False


In [ ]:
# Validate the environment and import the shared package implementation.
import shutil
import sys
import pandas as pd
from IPython.display import HTML, display
from rich.console import Console

PACKAGE_DIR = Path.cwd()
if not (PACKAGE_DIR / "utils").is_dir():
    raise RuntimeError("Run this notebook from the package directory.")
sys.path.insert(0, str(PACKAGE_DIR))
from utils.designer import HANDLE_SEQUENCES, design_allele_primers_for_genes

required_files = [CONFIG["gtf_path"], CONFIG["genome_fasta"], CONFIG["snp_vcf"]]
if CONFIG["blast_specificity"]:
    required_files.append(CONFIG["blast_database"] + ".nsq")
missing_files = [path for path in required_files if not Path(path).exists()]
missing_tools = [CONFIG[key] for key in ("samtools", "primer3_core") if not Path(CONFIG[key]).exists()]
if CONFIG["blast_specificity"] and not Path(CONFIG["blastn"]).exists():
    missing_tools.append(CONFIG["blastn"])
if missing_files or missing_tools:
    raise FileNotFoundError({"missing_files": missing_files, "missing_tools": missing_tools})
if CONFIG["handle"] not in HANDLE_SEQUENCES:
    raise ValueError(f"Unsupported handle: {CONFIG['handle']}")

Console().print(f"[green]Validated[/green] {len(GENES)} genes; output: {OUTPUT_DIR}")


In [ ]:
# Run the pipeline. Rich displays [Gene i/N] and SNP-target progress.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
batch_results = design_allele_primers_for_genes(
    gene_names=GENES,
    output_root=OUTPUT_DIR,
    config=CONFIG,
    show_progress=CONFIG["show_progress"],
)
batch_results


In [ ]:
# Load consolidated outputs.
summary_path = OUTPUT_DIR / "allele_primer_batch_summary.csv"
oligos_path = OUTPUT_DIR / "allele_primer_consolidated_top_oligos.csv"
report_path = OUTPUT_DIR / "allele_primer_consolidated_report.html"
batch_summary = pd.read_csv(summary_path)
top_oligos = pd.read_csv(oligos_path)
batch_summary


In [ ]:
# Verify order sequences and strand-independent cDNA/SNP constraints.
if not top_oligos.empty:
    assert set(top_oligos["Handle_Name"]) == {CONFIG["handle"]}
    assert (top_oligos["Final_Order_Sequence"] == top_oligos["Handle_Sequence"] + top_oligos["Primer_Insert_Seq"]).all()
    distances = top_oligos["Primer_To_SNP_Distance_Toward_PolyDT"]
    assert ((distances > 0) & (distances <= CONFIG["snp_window_toward_polydt"])).all()
    assert (~top_oligos["Primer_Overlaps_SNP"].astype(bool)).all()
    if CONFIG["blast_specificity"]:
        assert top_oligos["Primer_Specificity_Pass"].astype(bool).all()
print(f"QC passed for {len(top_oligos)} consolidated oligos.")


In [ ]:
# Review order-ready oligos and open the consolidated HTML report.
display_columns = [
    "Gene", "Transcript_ID", "Strand", "Insert_Rank", "Primer_Insert_Seq",
    "Handle_Name", "Final_Order_Sequence", "SNP_cDNA", "SNP_Genomic",
    "SNP_Distance_To_RNA_3p", "Primer_To_SNP_Distance_Toward_PolyDT",
    "Primer_Tm", "Primer_GC", "Primer_Specificity_Pass",
]
display(top_oligos[display_columns] if not top_oligos.empty else top_oligos)
display(HTML(f'<p><a href="{report_path.as_uri()}" target="_blank">Open consolidated HTML report</a></p><p>Output directory: <code>{OUTPUT_DIR}</code></p>'))
